# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar320250068-prog/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Author:** Omar Hindawy · **Week 1 assignment (ML-02)**

This notebook frames *one* provisional research question before any modelling. Per the
`framing-ml-problems` skill, a model is never the goal — a better **decision** is. Each section
below is filled in order, backed by real numbers from the starter dataset
(`data/raw/content_refresh_anonymized.csv`).

I loaded two repo skills for this task: `skills/framing-ml-problems/SKILL.md` (the framing method)
and `skills/flyrank/flyrank-data/SKILL.md` (the data gotchas). I can confirm the lane choice or
switch until the end of Week 4.


## 0. Setup

Locate the repo root so the notebook runs top-to-bottom locally (Jupyter/VS Code)
*and* on Colab, where it clones the repo first.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omar320250068-prog/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    if not os.path.isdir("data/raw"):
        for cand in (r"C:\Users\OMAR HINDAWY\flyrank-ml-internship", os.getcwd()):
            if os.path.isdir(os.path.join(cand, "data", "raw")):
                os.chdir(cand)
                break

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — run from the repo"
print("Starter data found. Ready.")


Working dir: C:\Users\OMAR HINDAWY\flyrank-ml-internship
Starter data found. Ready.


## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring** (a core lane, not freestyle).

**Why:** the starter data already shows that a real, position-adjusted opportunity exists and is
computable today. Among pages that are genuinely *visible* (≥ 500 impressions/90d and a recorded
average position of 1–20), **12,023 of 30,000 (40.1%)** qualify for review, and **25.2% of those
sit below half their own position bucket's median CTR**. CTR does not fall neatly with position in
this slice (correlation only **−0.125**), which tells me the *within-tier* comparison is the honest
lens and that a plain "low CTR = bad" rule would misfire. This lane answers a decision a content
editor actually makes every week ("which visible page is under-capturing clicks and is worth a
title/snippet review?"), the starter pipeline already gives me a transparent baseline and a
learned-model comparison to beat, and the data is dense enough (every row here has ≥ 1 impression,
100% have ≥ 1 session) that I am not fighting the sparsity that sinks the AI-referral direction
(only 6.4% of rows have any AI session).

I chose this over Lane 2 (refresh scoring) partly because the starter slice's stale-and-visible
stratum is nearly empty here (**17 rows, 0.1%**) — that lane needs the warehouse's time windows to
have enough signal, whereas the CTR lane is already well-populated in the small dataset I can run
today.


In [2]:
# Section 1 backing numbers — Lane 4 (CTR / Engagement) viability
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
visible["pos_bin"] = pd.cut(visible["avg_position"], bins=[0, 3, 10, 20],
                            labels=["top_3", "page_1", "striking"])

print("Starter slice:", len(df), "pages |", df["client_id"].nunique(), "clients")
print(f"Visible pages (impr>=500, position 1-20): {len(visible):,} "
      f"({len(visible)/len(df):.1%} of slice)")

# stale-and-visible stratum for Lane 2, to justify NOT picking Lane 2 on this slice alone
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale+visible pages (a Lane-2 signal): {stale_visible} "
      f"({stale_visible/len(df):.1%}) -> thin here, needs the warehouse")

# AI-referral direction sparsity, to justify NOT picking the freestyle AI lane
ai_rows = (df["ai_sessions_90d"] > 0).sum()
print(f"Rows with any AI session: {ai_rows:,} ({ai_rows/len(df):.1%}) -> sparse, EDA-only at best")


Starter slice: 30000 pages | 32 clients
Visible pages (impr>=500, position 1-20): 12,023 (40.1% of slice)
Stale+visible pages (a Lane-2 signal): 17 (0.1%) -> thin here, needs the warehouse
Rows with any AI session: 1,930 (6.4%) -> sparse, EDA-only at best


## 2. The question: decision, action, cost of a wrong call

**The one-paragraph frame** (from the `framing-ml-problems` skill):

> For a **content editor with limited weekly review capacity**, deciding **which visible pages to
> review first for a title/meta or intent-match fix**, we will build a **ranked opportunity score**
> from the starter dataset's observable search and engagement signals, **scoring pages on how far
> their CTR falls below the expected CTR for their own position tier**, measured by
> **precision@K on a held-out set of clients**. A wrong call costs **wasted editor hours on pages
> that were never going to convert, and — if the ranking is bad — missed pages that were quietly
> leaking clicks**. A plain rule isn't enough because the signal is **tangled**: position, intent,
> content type, and volume all interact, and CTR does not descend cleanly with position in this
> data (correlation only −0.125). We will claim only **observed / directional / decision-support**
> results.

**Decision.** Which pages a human reviewer looks at *first* this week. Not "will this page
recover" — that would need an experiment.

**Who acts, and how.** A content/SEO editor opens the top of the queue, reads the reason codes
(high impressions, weak CTR *for its position*, enough sessions), and rewrites a title, meta
description, or the intro to better match the query intent. The queue is a triage list; the human
makes the edit.

**Cost of a wrong call.**
- *False positive* (rank a fine page as an opportunity): wasted reviewer time; a needless edit can
  even hurt a page that was already working.
- *False negative* (miss a real click leak): a visible page keeps under-capturing clicks for
  another cycle — the more expensive error when volume is high.
- Because the reviewer's time is the scarce resource, the right metric is **precision@K**
  (of the top K pages I flag, how many are genuinely under-capturing), not raw accuracy.

**Why data/ML helps at all.** The pattern is real but messy: different position tiers have very
different normal CTRs, low-volume pages are noise, and intent shifts the baseline. A single
hand-written threshold applied to all pages would either flood the queue or miss most of it. That
is exactly the "real but too tangled to write by hand" condition where a learned or
tier-adjusted score earns its place.


In [3]:
# Section 2 backing numbers — size, cost asymmetry, and how much is at stake
# NOTE: these are OBSERVED aggregates from the starter slice, not causal estimates.
visible["exp_ctr_by_bin"] = visible["pos_bin"].astype(str).map(
    visible.groupby(visible["pos_bin"].astype(str))["ctr"].median()
)

under = visible[visible["ctr"] < 0.5 * visible["exp_ctr_by_bin"]].copy()

print("Median CTR by position bucket (context for the tier-adjusted comparison):")
print(visible.groupby(visible["pos_bin"].astype(str))["ctr"].median().round(3).to_string())

print(f"\nPages below HALF their bucket median CTR: {len(under):,} "
      f"({len(under)/len(visible):.1%} of visible pages)")
print(f"Combined impressions_90d on those pages: {int(under['impressions_90d'].sum()):,}")
print(f"Clicks they currently earn: {int(under['clicks_90d'].sum()):,}")

# Illustrative upside ONLY: if these pages matched their bucket median CTR, rough extra clicks.
under["extra_clicks_if_median"] = (
    under["impressions_90d"] * under["exp_ctr_by_bin"] / 100 - under["clicks_90d"]
).clip(lower=0)
print(f"Illustrative extra clicks if each hit its bucket median CTR: "
      f"{int(under['extra_clicks_if_median'].sum()):,}")
print("(Directional sizing of the opportunity — NOT a recovery promise; see section 4.)")


Median CTR by position bucket (context for the tier-adjusted comparison):
pos_bin
page_1      0.24
striking    0.17
top_3       0.20

Pages below HALF their bucket median CTR: 3,033 (25.2% of visible pages)
Combined impressions_90d on those pages: 21,501,374
Clicks they currently earn: 13,143
Illustrative extra clicks if each hit its bucket median CTR: 35,473
(Directional sizing of the opportunity — NOT a recovery promise; see section 4.)


## 3. Quick look at the data (2-3 real numbers)

Three real numbers from `data/raw/content_refresh_anonymized.csv` that make this lane look worth
the next 7 weeks:

1. **40.1% of the slice is reviewable and visible** — **12,023 of 30,000** pages have ≥ 500
   impressions/90d and a recorded average position of 1–20. The queue is neither empty nor
   everything.
2. **A genuine, position-adjusted gap exists** — **25.2% of those 12,023 visible pages** earn less
   than *half* the median CTR of their own position bucket. That is a sizable, targeted set, not
   noise.
3. **The naive signal is misleading, which is the point** — the raw correlation between average
   position and CTR across visible pages is only **−0.125**: position alone explains almost
   nothing here, so a better (tier-adjusted or learned) score has real room to help. For contrast,
   the starter pipeline's own learned model already beat its hand rule **~2.8×** at precision@50
   (0.68 vs 0.24) on the related decline task — evidence that learned ranking is worth pursuing on
   this slice.


In [4]:
# Section 3 backing numbers — the three headline figures (recomputed cleanly)
print("NUMBER 1 — reviewable & visible share")
print(f"  visible pages: {len(visible):,} of {len(df):,} = {len(visible)/len(df):.1%}")

print("\nNUMBER 2 — position-adjusted CTR gap")
print(f"  below half their bucket median CTR: {len(under):,} "
      f"= {len(under)/len(visible):.1%} of visible pages")

print("\nNUMBER 3 — the naive signal is weak, so a better score has room")
print(f"  corr(avg_position, ctr) among visible pages: {visible['avg_position'].corr(visible['ctr']):+.3f}")

print("\nRefresher — why learned ranking can help (from the starter pipeline run in notebooks/01):")
print("  hand rule  precision@50 = 0.24")
print("  random forest precision@50 = 0.68  (~2.8x the rule)")
print("  (library-version sensitive; the stable claim is the ~3x lift, per GUIDE.md FAQ)")


NUMBER 1 — reviewable & visible share
  visible pages: 12,023 of 30,000 = 40.1%

NUMBER 2 — position-adjusted CTR gap
  below half their bucket median CTR: 3,033 = 25.2% of visible pages

NUMBER 3 — the naive signal is weak, so a better score has room
  corr(avg_position, ctr) among visible pages: -0.125

Refresher — why learned ranking can help (from the starter pipeline run in notebooks/01):
  hand rule  precision@50 = 0.24
  random forest precision@50 = 0.68  (~2.8x the rule)
  (library-version sensitive; the stable claim is the ~3x lift, per GUIDE.md FAQ)


## 4. Careful words: what I can and can't claim

**What this work WILL be able to say (carefully):**
- **Observed / measured:** "In the starter slice, N pages sit below the expected CTR for their
  position tier." That is a description of the data, and it is reproducible.
- **Directional / decision-support:** "This score can rank visible pages so a reviewer looks at the
  most plausibly under-capturing ones first." A ranking is a prioritisation aid, not a promise.
- **Comparative:** "On the held-out clients, the learned/tier-adjusted score placed more genuinely
  under-capturing pages in the top K than the hand baseline did." A like-for-like comparison of two
  rankings under the same split and metric.

**What this work will NEVER claim:**
- **Not causal.** I will *not* claim "editing this page will recover clicks." Proving that needs an
  experiment (a controlled before/after or A/B design) that this data alone cannot give.
- **Not "predicting Google."** I will not claim to have discovered a Google algorithm factor.
  These are observed associations in one pseudonymized slice.
- **Not a guarantee on the warehouse.** The 30k-row slice is a teaching sample; a result here must
  be re-earned on the ~79M-row warehouse release with proper time-aware validation.
- **No leakage.** `trend_direction` and `trend_pct` are the label source and stay out of features;
  IDs (`content_id`, `client_id`) are used only for grouping and grouped train/test splits, never
  as features.
- **No private data.** Only pseudonymized IDs, aggregates, and charts; no client names, domains,
  URLs, or raw queries.

**Label discipline (for later weeks).** The starter's `is_declining_label` is a bucket computed
from the *current* window — a beginner proxy. For the capstone I intend to move to a
**future-window** target (e.g. features from a prior 90 days → CTR/position change over the next
30 days), defined and audited before modelling, with the feature window and target window never
overlapping.


In [5]:
# Section 4 backing checks — leakage discipline, ID hygiene, and that the metric is computable.
features_intended = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                     "position_tier", "main_intent", "content_type",
                     "content_age_days", "days_since_last_update",
                     "sessions_90d", "engagement_rate", "scroll_rate"]

leak_cols = {"trend_direction", "trend_pct", "is_declining_label"}
id_cols = {"content_id", "client_id"}

bad = [c for c in features_intended if c in leak_cols or c in id_cols]
print("Leakage/ID columns wrongly included as features:", bad or "none — good")

print("\nMetric sanity — is precision@K computable today, before any model?")
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Transparent first baseline: rank by how far actual CTR sits below the bucket's expected CTR.
visible["baseline_opportunity"] = (visible["exp_ctr_by_bin"] - visible["ctr"]) \
                                  * np.log1p(visible["impressions_90d"])

# A DIFFERENT, independent stand-in label: the target will change in later weeks, but here we
# only need *some* binary outcome to prove the metric computes end to end.
tier_q1 = visible.groupby(visible["pos_bin"].astype(str))["ctr"].transform(lambda s: s.quantile(0.25))
visible["is_under"] = (visible["ctr"] < tier_q1).astype(int)

print(f"  stand-in label base rate: {visible['is_under'].mean():.3f}")
for k in (20, 50):
    print(f"  hand baseline precision@{k}: "
          f"{precision_at_k(visible['baseline_opportunity'], visible['is_under'], k):.3f}")

print("""
Honest note: precision reads 1.000 here ON PURPOSE — both the baseline score and the stand-in
label are built from CTR-vs-tier, so the ranking trivially contains the label. This is the
CIRCULAR RESULT the lane guide warns about, shown deliberately. It is NOT evidence that the
hand rule is good. The real job of later weeks is to define a leakage-safe, future-window
target, rebuild an honest baseline, and let a model earn a channel that this rule cannot.
The point of this cell is narrower: the metric exists and computes today.
""")


Leakage/ID columns wrongly included as features: none — good

Metric sanity — is precision@K computable today, before any model?
  stand-in label base rate: 0.242
  hand baseline precision@20: 1.000
  hand baseline precision@50: 1.000

Honest note: precision reads 1.000 here ON PURPOSE — both the baseline score and the stand-in
label are built from CTR-vs-tier, so the ranking trivially contains the label. This is the
CIRCULAR RESULT the lane guide warns about, shown deliberately. It is NOT evidence that the
hand rule is good. The real job of later weeks is to define a leakage-safe, future-window
target, rebuild an honest baseline, and let a model earn a channel that this rule cannot.
The point of this cell is narrower: the metric exists and computes today.



## Self-check

- [x] Every section above is filled — markdown thinking **and** the code that backs it (sections 1–4)
- [x] The notebook runs top to bottom with no errors (executed and saved with outputs)
- [x] No client names, URLs, or private queries anywhere — only the anonymized starter slice, aggregates, and pseudonymized IDs
- [x] Claims use careful words: observed, measured, directional, decision-support — no causal or "Google algorithm" claims
- [x] Committed to my repo under `work/notebooks/`

**Provisional lane:** Lane 4 — CTR / Engagement Opportunity Scoring.
**Change window:** final choice can confirm or change until end of Week 4.
